# Import libraries

import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Load your dataset (CSV)

In [ ]:
df = pd.read_csv("/content/sample_data/final_dataset_cleaned.csv")   # upload via Colab file upload

# Check first rows
print(df.head())

# Create embeddings

In [ ]:
# Load embedding model
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Create embeddings for questions
question_embeddings = embedder.encode(df["question"].tolist(), convert_to_numpy=True)

# Dimension of embeddings
embedding_dim = question_embeddings.shape[1]


# Setup FAISS Index

In [ ]:
# Create FAISS index
index = faiss.IndexFlatL2(embedding_dim)

# Add vectors to FAISS
index.add(question_embeddings)

print("Total documents in FAISS:", index.ntotal)


# Retriever Function

In [ ]:
def retrieve(query, top_k=3):
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    for idx in indices[0]:
        results.append(df.iloc[idx]["answer"])
    return results


# Load Generator Model

generator = pipeline("text2text-generation", model="/content/sample_data/flan_t5_lora_medquad")

# Full RAG Pipeline

def rag_pipeline(query):
    # Step 1: Retrieve context
    context_answers = retrieve(query, top_k=3)
    context = " ".join(context_answers)

    # Step 2: Generate final answer
    prompt = f"Answer the question using the following context:\n\nContext: {context}\n\nQuestion: {query}\nAnswer:"
    response = generator(prompt, max_length=200, do_sample=True)[0]['generated_text']

    return response

# Test the Bot

user_query = "are certain people at risk of getting vancomycinresistant enterococci?"
print("User:", user_query)
print("Bot:", rag_pipeline(user_query))